# HelloML — Handwritten Digit OCR (DIDA)

Multi-model OCR with **train / load** modes so you can skip retraining.

| Mode | What it does |
|------|----------------|
| `RUN_MODE = "train"` | Load DIDA → preprocess → GridSearchCV → evaluate → **save** `artifacts/*.joblib` |
| `RUN_MODE = "load"` | **Load** artifacts → evaluate / plot (no training) |

Helpers live in `helloml_pipeline.py` (same folder as this notebook).


## 0–1. Setup & config


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from helloml_pipeline import (
    load_dataset, binarize_center_resize, normalize_flatten_split,
    get_experiment_setup, SCORING_METRICS,
    save_artifacts, load_artifacts, artifacts_exist, ARTIFACT_KEYS,
)

# ============================================================
# CONFIG
# ============================================================
RUN_MODE = "train"          # "train" | "load"
ARTIFACTS_DIR = Path.cwd() / "artifacts"
FILES_PER_FOLDER = 1000     # e.g. 200 for a quick dry-run
# ============================================================

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_MODE = {RUN_MODE}")
print(f"ARTIFACTS_DIR = {ARTIFACTS_DIR}")


## 2–4. Data (train or load)


In [ ]:
if RUN_MODE == "load":
    if not artifacts_exist(ARTIFACTS_DIR):
        raise FileNotFoundError(
            f"No artifacts in {ARTIFACTS_DIR}. Run once with RUN_MODE='train'."
        )
    print("Loading data artifacts...")
    load_artifacts(ARTIFACTS_DIR, globals(), keys=[
        "X_train", "X_test", "y_train", "y_test", "X_proc", "y"
    ])
    X_raw = None
elif RUN_MODE == "train":
    root = Path.cwd() / "DIDA"
    if not root.exists():
        raise FileNotFoundError(f"DIDA not found in {Path.cwd()}")
    X_raw, y = load_dataset(root, files_per_folder=FILES_PER_FOLDER)
    X_proc = binarize_center_resize(X_raw)
    X_train, X_test, y_train, y_test = normalize_flatten_split(X_proc, y)
else:
    raise ValueError("RUN_MODE must be 'train' or 'load'")


## 5. Preview samples


In [ ]:
if X_proc is not None:
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for digit, ax in enumerate(axes.ravel()):
        idxs = np.where(y == digit)[0]
        ax.imshow(X_proc[idxs[0]], cmap="gray")
        ax.set_title(f"{digit}")
        ax.axis("off")
    plt.suptitle("Preprocessed samples", fontweight="bold")
    plt.tight_layout()
    plt.show()


## 6–7. Models & GridSearchCV (or load models)


In [ ]:
setup = get_experiment_setup()
scoring_metrics = SCORING_METRICS

if RUN_MODE == "load":
    print("Loading model artifacts (skip training)...")
    load_artifacts(ARTIFACTS_DIR, globals(), keys=[
        "best_estimators", "grids", "df_cv_all", "df_cv_best", "df_test"
    ])
    display(df_cv_best)
else:
    best_estimators, grids, cv_rows, cv_best_rows = {}, {}, [], []
    print("=" * 60)
    print("PHASE 1: GridSearchCV")
    print("=" * 60)
    for name, cfg in setup.items():
        print(f"\n>>> {name}")
        grid = GridSearchCV(
            estimator=cfg["model"],
            param_grid=cfg["params"],
            cv=5,
            scoring=scoring_metrics,
            refit="f1",
            n_jobs=-1,
            return_train_score=False,
            verbose=1,
        )
        t0 = time.time()
        grid.fit(X_train, y_train)
        print(f"    done in {time.time()-t0:.1f}s | best: {grid.best_params_}")
        best_estimators[name] = grid.best_estimator_
        grids[name] = grid
        res, bi = grid.cv_results_, grid.best_index_
        cv_best_rows.append({
            "Model": name,
            "Best Params": grid.best_params_,
            "Mean Accuracy": res["mean_test_accuracy"][bi],
            "Mean Precision": res["mean_test_precision"][bi],
            "Mean Recall": res["mean_test_recall"][bi],
            "Mean F1": res["mean_test_f1"][bi],
            "Std F1": res["std_test_f1"][bi],
            "Mean Fit Time (s)": res["mean_fit_time"][bi],
        })
        for i in range(len(res["params"])):
            cv_rows.append({
                "Model": name,
                "Params": res["params"][i],
                "Mean Accuracy": res["mean_test_accuracy"][i],
                "Mean Precision": res["mean_test_precision"][i],
                "Mean Recall": res["mean_test_recall"][i],
                "Mean F1": res["mean_test_f1"][i],
                "Std F1": res["std_test_f1"][i],
                "Mean Fit Time (s)": res["mean_fit_time"][i],
            })
    df_cv_all = pd.DataFrame(cv_rows)
    df_cv_best = pd.DataFrame(cv_best_rows).sort_values("Mean F1", ascending=False)
    display(df_cv_best)
    print("\nSaving artifacts...")
    save_artifacts(ARTIFACTS_DIR, globals())


## 8. CV confusion matrices


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, model) in zip(axes.ravel(), best_estimators.items()):
    y_pred_cv = cross_val_predict(model, X_train, y_train, cv=5)
    cm = confusion_matrix(y_train, y_pred_cv)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(f"CV CM — {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()


## 9. Test evaluation


In [ ]:
print("=" * 60)
print("PHASE 2: Test set")
print("=" * 60)
test_rows = []
target_names = [f"Digit {i}" for i in range(10)]
for name, model in best_estimators.items():
    print(f"\n--- {name} ---")
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Test Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=target_names, digits=3))
    test_rows.append({"Model": name, "Test Accuracy": acc})
df_test = pd.DataFrame(test_rows).sort_values("Test Accuracy", ascending=False)
display(df_test)
if RUN_MODE == "train":
    joblib.dump(df_test, ARTIFACTS_DIR / "df_test.joblib")


## 10. Charts


In [ ]:
df_plot = df_cv_best.merge(df_test, on="Model")
colors = ["#7e57c2", "#26a69a", "#42a5f5", "#ef5350"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title in [
    (axes[0], "Mean Accuracy", "Mean CV Accuracy"),
    (axes[1], "Test Accuracy", "Test Accuracy"),
]:
    bars = ax.bar(df_plot["Model"], df_plot[col], color=colors[:len(df_plot)])
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    for b in bars:
        h = b.get_height()
        ax.annotate(f"{h:.1%}", (b.get_x()+b.get_width()/2, h),
                    ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()


## 11. Manual save / load


In [ ]:
# save_artifacts(ARTIFACTS_DIR, globals())
# load_artifacts(ARTIFACTS_DIR, globals())

print("Artifacts:", ARTIFACTS_DIR)
for p in sorted(ARTIFACTS_DIR.glob("*.joblib")):
    print(f"  {p.name:30s}  {p.stat().st_size/1024/1024:6.2f} MB")


## Workflow

```text
1st time:  RUN_MODE = "train"  → writes artifacts/*.joblib
Later:     RUN_MODE = "load"   → no GridSearchCV, just plots & reports
```
